# Haas Japanese Colab ノートブック

このノートブックは、Third Intelligenceの **Haas-ja-v1.0** を使用して、日本語の音声・動画ファイルを高速に文字起こしします。

## 特長

- **無料:** Google Colabの無料GPUランタイムで実行できます。
- **日本語向け音声認識モデル:** Third Intelligenceの `third-intelligence/Haas-ja-v1.0` を使用します。日本語音声を文字に変換するための約0.6Bパラメータのモデルです。
- **句読点を出力:** 認識結果には、モデルが推定した「、」「。」などの句読点が含まれます。
- **時刻情報・字幕:** `subtitle` では音声分割区間の概算時刻を利用してSRTを生成します。文字・単語ごとの正確な時刻は取得できません。`none` では時刻情報を付けません。
- **長時間ファイル対応:** 長い音声・動画は、内部で短い音声に分割して順番に文字起こしします。
- **Google Drive対応:** 直接アップロードまたはGoogle Drive内のファイルを処理できます。
- **複数形式:** TXT / Markdown / JSON / CSV / XLSX / SRTを出力し、最後にZIPへまとめます。

## 使い方

1. **1. 設定**で INPUT_MODE を選びます。必要に応じてその他の設定も変更します。
2. **Runtime → Run all**を選択します。
3. `INPUT_MODE = "upload"` の場合は、**2. ファイル選択 / Google Driveマウント**でファイルを選択します。
4. 処理終了後、出力ZIPが自動ダウンロードされます。

## 備考

- GPUが利用できない場合、文字起こし開始前に停止します。
- 処理には時間がかかります。目安として、90 分の音声・動画ファイルの文字起こしには数分程度かかることがあります。初回実行時は、NeMoと約1.23GBのモデルをColab上で取得するため、文字起こし開始まで時間がかかる場合があります。ファイルを直接アップロードする場合は、アップロード時間も必要です。
- 長時間音声は内部で分割して処理するため、分割位置の前後で文脈が途切れ、認識結果に影響する場合があります。`CHUNK_SECONDS` を大きくすると分割回数は減りますが、1回の処理が重くなります。小さくすると1回の処理は軽くなりますが、分割位置が増えます。
- Third Intelligence公開モデルのライセンスは **Apache-2.0** です。


In [ ]:
# @title 1. 設定
# ============================================================
# 1. 設定
# ============================================================
import os
os.environ["USE_TF"] = "0"  # Set before importing Transformers.
from pathlib import Path

# @markdown ## 音声・動画ファイル
# @markdown - `INPUT_MODE`: ファイルを直接アップロードするときは `upload`を、Google Driveに保存されているファイルを取得するときは `google_drive`を使用します。
INPUT_MODE = "upload"  # @param ["upload", "google_drive"]

# @markdown ### Google Driveのファイルを取得する
# @markdown これらの設定は、`INPUT_MODE == "google_drive"`の場合にのみ使用されます。
# @markdown - `GOOGLE_DRIVE_ROOT`: 通常、これは `/content/drive/MyDrive`に設定しておいてください。
# @markdown - `GOOGLE_DRIVE_SUBPATH`: MyDrive 下のファイルまたはフォルダのパス。最上位の MyDrive フォルダを使用する場合は、空白のままにします。
# @markdown - `RECURSIVE`: ドライブの入力パスがフォルダの場合は、サブフォルダも検索します。
# @markdown - `FILENAME_CONTAINS`: オプションのファイル名の部分文字列フィルタ。
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
GOOGLE_DRIVE_SUBPATH = ""  # @param {type:"string"}
RECURSIVE = False  # @param {type:"boolean"}
FILENAME_CONTAINS = ""  # @param {type:"string"}

# @markdown ## デコーダー
# @markdown - Haas-ja-v1.0はTDT方式のみです。CTC方式には切り替えられません。
MODEL_ID = "third-intelligence/Haas-ja-v1.0"

# @markdown ## タイムスタンプ
# @markdown - `TIMESTAMP_MODE`: `none` は時刻情報なし、`subtitle` は音声分割区間の概算時刻でSRTを生成します（単語単位の時刻ではありません）。
TIMESTAMP_MODE = "subtitle"  # @param ["subtitle", "none"]

# @markdown ## 長時間音声
# @markdown - `CHUNK_SECONDS`: 長時間ファイルを内部で何秒ずつに分けて処理するかを指定します。大きくすると分割回数が減って前後の文脈を保ちやすくなる一方、1回の処理が重くなりGPUメモリ使用量も増えやすくなります。小さくすると1回の処理は軽くなりますが、分割位置が増えるため、その前後で認識が途切れる可能性があります。通常は既定値のままで構いません。利用者が指定する入力ファイル自体は分割されません。
CHUNK_SECONDS = 28.0  # @param {type:"number"}
INFERENCE_BATCH_SIZE = 1

# 字幕表示（内部設定）
SUBTITLE_MAX_LINE_CHARS = 20

# @markdown ## 出力
# @markdown - `SAVE_OUTPUTS_TO_GOOGLE_DRIVE`: `True` の場合、入力方法にかかわらず出力ZIPを Google Driveに保存します。
# @markdown - `AUTO_DOWNLOAD_ZIP`: 最後に ZIPアーカイブを 1 つ作成してダウンロードします。
# - 出力形式は以下のブール値によって制御されます。
SAVE_OUTPUTS_TO_GOOGLE_DRIVE = False  # @param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True  # @param {type:"boolean"}
SAVE_TXT = True
SAVE_MD = True
SAVE_JSON = True
SAVE_CSV = True
SAVE_XLSX = True
SAVE_SRT = True
INCLUDE_TIMESTAMPS_IN_TXT = True

# Advanced
REQUIRE_CUDA = True
UPLOAD_DIR = "/content/haas_ja_uploads"
WORK_DIR = "/content/haas_ja_work"
OUTPUT_DIR = "/content/haas_ja_outputs"
ZIP_FILE_NAME = "haas_ja_outputs.zip"

SUPPORTED_EXTS = {
    ".aac", ".avi", ".flac", ".m4a", ".m4v", ".mkv", ".mov",
    ".mp3", ".mp4", ".ogg", ".opus", ".wav", ".webm", ".wma"
}

RETURN_TIMESTAMPS = TIMESTAMP_MODE == "subtitle"

if TIMESTAMP_MODE not in {"subtitle", "none"}:
    raise ValueError("TIMESTAMP_MODE は subtitle / none を指定してください。")
if not 0 < CHUNK_SECONDS <= 28:
    raise ValueError("CHUNK_SECONDS は0より大きく28以下にしてください。")

for p in [UPLOAD_DIR, WORK_DIR, OUTPUT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("設定を読み込みました。")


In [ ]:
# @title 2. ファイル選択 / Google Driveマウント
# ============================================================
# 2. ファイル選択 / Google Driveマウント
# ============================================================
from pathlib import Path
import shutil

def collect_supported_files(root: Path):
    if root.is_file():
        files = [root]
    elif root.is_dir():
        pattern = "**/*" if RECURSIVE else "*"
        files = [p for p in root.glob(pattern) if p.is_file()]
    else:
        raise FileNotFoundError(f"入力パスが存在しません: {root}")
    files = [p for p in files if p.suffix.lower() in SUPPORTED_EXTS]
    if FILENAME_CONTAINS:
        files = [p for p in files if FILENAME_CONTAINS in p.name]
    return sorted(files)

selected_files = []

if INPUT_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    upload_dir = Path(UPLOAD_DIR)
    for name, data in uploaded.items():
        target = upload_dir / Path(name).name
        target.write_bytes(data)
        if target.suffix.lower() in SUPPORTED_EXTS:
            selected_files.append(target)
elif INPUT_MODE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path(GOOGLE_DRIVE_ROOT)
    if GOOGLE_DRIVE_SUBPATH:
        root = root / GOOGLE_DRIVE_SUBPATH
    selected_files = collect_supported_files(root)
else:
    raise ValueError("INPUT_MODE は upload または google_drive を指定してください。")

if not selected_files:
    raise RuntimeError("対応する音声・動画ファイルが選択されていません。")

print("選択されたファイル:")
for p in selected_files:
    print(" -", p)


In [ ]:
# @title 3. GPUのチェックと依存関係のインストール
# ============================================================
# 3. GPUのチェックと依存関係のインストール
# ============================================================
import os, shutil, subprocess, sys
from pathlib import Path

if os.environ.get("USE_TF") != "0":
    raise RuntimeError("Run section 1 first in a fresh runtime (USE_TF=0 must be set before Transformers imports).")

import torch

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("GPUが利用できません。Colabで「ランタイム → ランタイムのタイプを変更 → GPU」を選択してください。")

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU compute capability:", torch.cuda.get_device_capability(0))
print("BF16対応:", torch.cuda.is_bf16_supported())

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.check_call(["apt-get", "-qq", "update"])
    subprocess.check_call(["apt-get", "-qq", "install", "-y", "ffmpeg"])

# Official Haas-ASR source pinned to a reviewed upstream commit.
# Keep the installer and the preflight on the same Transformers version.
TRANSFORMERS_VERSION = "4.57.6"
install_command = [
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "off",
    "haas-asr @ git+https://github.com/third-intelligence/Haas-ASR.git@acc614f68769dcbfae9dba96b48439c4a66295af",
    f"transformers=={TRANSFORMERS_VERSION}", "pandas", "openpyxl",
]
# Preserve the real pip error rather than a quiet CalledProcessError.
installation = subprocess.run(
    install_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
log_path = Path("/content/haas_ja_pip_install.log")
log_path.write_text(installation.stdout or "", encoding="utf-8")
if installation.returncode != 0:
    print(f"pip install failed (exit {installation.returncode}). Full log: {log_path}")
    print("\n".join((installation.stdout or "").splitlines()[-120:]))
    raise RuntimeError(f"Dependency installation failed; full pip log: {log_path}")
print("依存関係のインストールが完了しました。")
print("Full pip log:", log_path)

# Check the decoder in a fresh Python interpreter before downloading the model.
preflight = (
    "import importlib.metadata, os, sys; "
    "assert os.environ.get('USE_TF') == '0', 'USE_TF must be 0'; "
    f"assert importlib.metadata.version('transformers') == {TRANSFORMERS_VERSION!r}; "
    "from transformers.utils import is_tf_available; "
    "assert not is_tf_available(), 'TensorFlow import must be disabled'; "
    "import haas_asr; from haas_asr._fla import check_fla_version; check_fla_version(); "
    "from fla.layers.kda import KimiDeltaAttention; "
    "assert 'tensorflow' not in sys.modules, 'TensorFlow loaded unexpectedly'"
)
subprocess.check_call([sys.executable, "-c", preflight])
print("Decoder import check passed. Continue to model loading.")


In [ ]:
# @title 4. ヘルパー関数
# ============================================================
# 4. ヘルパー関数
# ============================================================
from pathlib import Path
import csv, json, math, re, shutil, subprocess
import pandas as pd

ASCII_WORD = re.compile(r"[A-Za-z0-9]")

def safe_stem(path: Path) -> str:
    name = re.sub(r'[\\/:*?"<>|]+', "_", path.stem).strip()
    return name or "transcription"

def run_command(cmd):
    cp = subprocess.run(cmd, capture_output=True, text=True)
    if cp.returncode != 0:
        raise RuntimeError((cp.stderr or cp.stdout or "コマンド実行に失敗しました。").strip())
    return cp

def convert_to_mono_16k_wav(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    run_command([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(src), "-vn", "-ac", "1", "-ar", "16000",
        "-c:a", "pcm_s16le", str(dst)
    ])
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError(f"音声の抽出に失敗しました: {src}")

def join_text_parts(parts):
    result = ""
    for part in parts:
        part = (part or "").strip()
        if not part:
            continue
        if result and ASCII_WORD.search(result[-1]) and ASCII_WORD.search(part[0]):
            result += " "
        result += part
    return result

def estimated_segment(text: str, start: float, end: float, chunk_index: int):
    """Approximate subtitle span: chunk boundaries, not forced alignment."""
    return {
        "type": "segment", "text": text, "start": start, "end": end,
        "duration": end - start, "chunk_index": chunk_index,
        "timing_source": "approximate_chunk_bounds",
    }

def build_subtitle_cues(all_ts):
    if TIMESTAMP_MODE != "subtitle":
        return [], "none"
    cues = [
        {"start": x["start"], "end": x["end"], "text": x["text"]}
        for x in all_ts["segment"] if x["text"].strip()
    ]
    return cues, "approximate_chunk_bounds"

def format_srt_timestamp(seconds):
    ms = max(0, int(round(float(seconds) * 1000)))
    h, rem = divmod(ms, 3600000)
    m, rem = divmod(rem, 60000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def wrap_cue_text(text, max_line_chars):
    text = (text or "").strip()
    if not text or len(text) <= max_line_chars:
        return text
    target = len(text) // 2
    candidates = [i + 1 for i, ch in enumerate(text) if ch in "、。！？!?，,．."]
    candidates = [i for i in candidates if 1 <= i < len(text)]
    split = min(candidates, key=lambda i: abs(i - target)) if candidates else min(max_line_chars, len(text)-1)
    return text[:split].rstrip() + "\n" + text[split:].lstrip()

def cues_to_srt(cues):
    blocks = []
    for i, cue in enumerate(cues, 1):
        blocks.append(
            f"{i}\n"
            f"{format_srt_timestamp(cue['start'])} --> {format_srt_timestamp(cue['end'])}\n"
            f"{wrap_cue_text(cue['text'], SUBTITLE_MAX_LINE_CHARS)}"
        )
    return "\n\n".join(blocks) + ("\n" if blocks else "")

def flatten_timestamps(all_ts):
    rows = []
    for kind in ("segment", "word", "char"):
        rows.extend(all_ts[kind])
    return rows

def write_outputs(source_path, payload, out_dir):
    stem = safe_stem(source_path)
    paths = []
    text = payload["text"]
    cues = payload["subtitle_cues"]

    if SAVE_TXT:
        p = out_dir / f"{stem}.txt"
        parts = [text]
        if INCLUDE_TIMESTAMPS_IN_TXT and cues:
            parts += ["", "=== Timestamps ==="]
            parts += [
                f"[{format_srt_timestamp(c['start'])} --> {format_srt_timestamp(c['end'])}] {c['text']}"
                for c in cues
            ]
        p.write_text("\n".join(parts) + "\n", encoding="utf-8")
        paths.append(p)

    if SAVE_MD:
        p = out_dir / f"{stem}.md"
        body = f"# {source_path.name}\n\n{text}\n"
        if cues:
            body += "\n## Timestamps\n\n"
            body += "\n".join(
                f"- `{format_srt_timestamp(c['start'])} --> {format_srt_timestamp(c['end'])}` {c['text']}"
                for c in cues
            )
            body += "\n"
        p.write_text(body, encoding="utf-8")
        paths.append(p)

    if SAVE_JSON:
        p = out_dir / f"{stem}.json"
        p.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        paths.append(p)

    rows = flatten_timestamps(payload["timestamps"])
    if SAVE_CSV:
        p = out_dir / f"{stem}.csv"
        pd.DataFrame(rows, columns=["type","text","start","end","duration","chunk_index"]).to_csv(
            p, index=False, encoding="utf-8-sig"
        )
        paths.append(p)

    if SAVE_XLSX:
        p = out_dir / f"{stem}.xlsx"
        with pd.ExcelWriter(p, engine="openpyxl") as writer:
            for kind in ("segment", "word", "char"):
                pd.DataFrame(payload["timestamps"][kind]).to_excel(
                    writer, sheet_name=kind + "s", index=False
                )
            pd.DataFrame(payload["chunks"]).to_excel(writer, sheet_name="chunks", index=False)
            pd.DataFrame(cues).to_excel(writer, sheet_name="subtitles", index=False)
        paths.append(p)

    if SAVE_SRT and TIMESTAMP_MODE == "subtitle" and cues:
        p = out_dir / f"{stem}.srt"
        p.write_text(cues_to_srt(cues), encoding="utf-8")
        paths.append(p)

    return paths


In [ ]:
# @title 5. Haas Japaneseをロード
# ============================================================
# 5. Haas Japaneseをロード
# ============================================================
import torch
import haas_asr

torch.set_float32_matmul_precision("high")
# Use CUDA compute capability: T4 (SM 7.5) has no native BF16 even if a support flag says True.
model_dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("モデルを読み込み中:", MODEL_ID, "dtype:", model_dtype)
asr_model = haas_asr.load_model(MODEL_ID, device="cuda", dtype=model_dtype)
print("モデルを読み込みました。")


In [ ]:
# @title 6. 文字起こし
# ============================================================
# 6. 文字起こし
# ============================================================
import json, traceback
from pathlib import Path
import haas_asr

output_root = Path(OUTPUT_DIR)
output_root.mkdir(parents=True, exist_ok=True)
manifest = []

for source_path in selected_files:
    source_path = Path(source_path)
    print("\n=== 処理開始:", source_path.name, "===")
    item_work = Path(WORK_DIR) / safe_stem(source_path)
    item_work.mkdir(parents=True, exist_ok=True)
    wav_path = item_work / "source_16k_mono.wav"
    record = {"source": source_path.name, "status": "running"}

    try:
        convert_to_mono_16k_wav(source_path, wav_path)
        audio = haas_asr.load_audio(str(wav_path))
        duration = len(audio) / haas_asr.SAMPLE_RATE
        bounds = haas_asr.split_at_quiet_points(
            audio, chunk_seconds=CHUNK_SECONDS,
            search_seconds=2.0,
        )
        if not bounds:
            raise RuntimeError("Haasから文字起こし結果が返されませんでした。")

        chunk_records = []
        chunk_texts = []
        all_ts = {"char": [], "word": [], "segment": []}
        for idx, (first, last) in enumerate(bounds):
            start = first / haas_asr.SAMPLE_RATE
            end = last / haas_asr.SAMPLE_RATE
            print(f"  chunk {idx+1}/{len(bounds)}: {start:.1f}s - {end:.1f}s")
            # Inference is owned by the official Haas API; no older NeMo
            # TDT/CTC decoder, CUDA-graph patch or timestamp flags survive.
            text = haas_asr.transcribe(
                asr_model, [audio[first:last]],
                batch_size=INFERENCE_BATCH_SIZE,
            )[0]
            chunk_texts.append(text)
            if RETURN_TIMESTAMPS and text.strip():
                all_ts["segment"].append(estimated_segment(text, start, end, idx))
            chunk_records.append({
                "index": idx, "start": start, "end": end,
                "duration": end - start, "text": text,
                "timestamp_counts": {
                    "char": 0, "word": 0,
                    "segment": int(bool(RETURN_TIMESTAMPS and text.strip())),
                },
            })

        full_text = join_text_parts(chunk_texts)
        cues, cue_source = build_subtitle_cues(all_ts)
        payload = {
            "source": source_path.name,
            "model": MODEL_ID,
            "decoder": "TDT",
            "timestamp_mode": TIMESTAMP_MODE,
            "timestamp_precision": "approximate_chunk_bounds" if RETURN_TIMESTAMPS else "none",
            "duration_seconds": duration,
            "chunk_seconds": CHUNK_SECONDS,
            "text": full_text,
            "chunks": chunk_records,
            "timestamps": all_ts,
            "subtitle_source": cue_source,
            "subtitle_cues": cues,
            "diagnostics": {
                "chunk_count": len(bounds),
                "characters": len(full_text),
                "char_timestamps": 0,
                "word_timestamps": 0,
                "segment_timestamps": len(all_ts["segment"]),
                "subtitle_cues": len(cues),
                "subtitle_max_duration": max(
                    (c["end"] - c["start"] for c in cues), default=0.0
                ),
                "subtitle_max_chars": max((len(c["text"]) for c in cues), default=0),
                "timestamps_monotonic": all(
                    rows[i]["start"] >= rows[i - 1]["start"]
                    for rows in all_ts.values() for i in range(1, len(rows))
                ),
            },
        }
        paths = write_outputs(source_path, payload, output_root)
        print("  文字数:", len(full_text))
        print("  native segment timestamps:", 0)
        print("  word timestamps:", 0)
        print("  approximate subtitle cues:", len(cues))
        record.update({
            "status": "ok", "duration_seconds": duration,
            "characters": len(full_text), "chunks": len(bounds),
            "char_timestamps": 0, "word_timestamps": 0,
            "segment_timestamps": len(all_ts["segment"]),
            "subtitle_cues": len(cues), "subtitle_source": cue_source,
            "outputs": [p.name for p in paths],
        })
    except Exception as exc:
        record.update({
            "status": "error", "error_type": type(exc).__name__,
            "error": str(exc), "traceback": traceback.format_exc(),
        })
        print(record["traceback"])
    manifest.append(record)

manifest_path = output_root / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("\n処理が完了しました。")
print("Manifest:", manifest_path)


In [ ]:
# @title 7. ZIP保存・ダウンロード
# ============================================================
# 7. ZIP保存・ダウンロード
# ============================================================
from pathlib import Path
import shutil

output_root = Path(OUTPUT_DIR)
zip_base = Path("/content") / Path(ZIP_FILE_NAME).stem
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=output_root))
print("ZIP:", zip_path)

if SAVE_OUTPUTS_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_out = Path(GOOGLE_DRIVE_ROOT) / "Haas_Japanese_ASR_outputs"
    drive_out.mkdir(parents=True, exist_ok=True)
    target = drive_out / zip_path.name
    shutil.copy2(zip_path, target)
    print("Google Driveに保存しました:", target)

if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
